# Branch 1: VGG-BN on CIFAR-10

Convolutional branch implementation using a 7-layer VGG network with Batch Normalization and dropout.

**Dataset Splits (CIFAR-10, seed 42)**:
- `train` (40k): Model parameter optimization with random crop and flip.
- `monitor` (5k): Unaugmented validation set for checkpoint selection.
- `val` (5k): Held-out split for downstream Reinforcement Learning fusion calibration.
- `test` (10k): Final evaluation split.

Outputs exported for fusion:
- `cnn_val_export.pt`: Embeddings and softmax predictions on `val`.
- `cnn_test_export.pt`: Embeddings and softmax predictions on `test`.


In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as transforms

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
NUM_CLASSES = 10
EMBED_DIM = 128
BATCH_SIZE = 128
NUM_EPOCHS = 50
LR = 1e-3
WEIGHT_DECAY = 1e-2
SUBSET_SIZE = None

print(f"Device: {device} | Epochs: {NUM_EPOCHS} | Batch size: {BATCH_SIZE}")


## 1. Data Preparation


In [ ]:
class TransformedSubset(Dataset):
    def __init__(self, dataset, indices, transform):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, target = self.dataset[self.indices[idx]]
        if self.transform is not None:
            img = self.transform(img)
        return img, target

norm_mean = (0.4914, 0.4822, 0.4465)
norm_std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std),
])

raw_cifar_train = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
raw_cifar_test = torchvision.datasets.CIFAR10(root="./data", train=False, download=True)

# 40k train / 5k monitor / 5k val (RL calibration) / 10k test
g = torch.Generator().manual_seed(42)
indices = torch.randperm(len(raw_cifar_train), generator=g).tolist()

if SUBSET_SIZE is not None:
    n_tr, n_mo, n_va = int(SUBSET_SIZE * 0.8), int(SUBSET_SIZE * 0.1), int(SUBSET_SIZE * 0.1)
else:
    n_tr, n_mo, n_va = 40000, 5000, 5000

train_idx = indices[:n_tr]
monitor_idx = indices[n_tr:n_tr + n_mo]
val_idx = indices[n_tr + n_mo:n_tr + n_mo + n_va]

train_ds = TransformedSubset(raw_cifar_train, train_idx, train_transform)
monitor_ds = TransformedSubset(raw_cifar_train, monitor_idx, eval_transform)
val_ds = TransformedSubset(raw_cifar_train, val_idx, eval_transform)
test_ds = TransformedSubset(raw_cifar_test, list(range(len(raw_cifar_test))), eval_transform)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=(device.type == "cuda"))
monitor_dl = DataLoader(monitor_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2)

print(f"Splits: Train={len(train_ds)}, Monitor={len(monitor_ds)}, RL Val={len(val_ds)}, Test={len(test_ds)}")


## 2. VGG-BN Architecture


In [ ]:
class VGGCIFAR(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, embed_dim=EMBED_DIM):
        super().__init__()
        self.features = nn.Sequential(
            # Stage 1: 32x32 -> 16x16
            nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.1),

            # Stage 2: 16x16 -> 8x8
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.2),

            # Stage 3: 8x8 -> 4x4
            nn.Conv2d(128, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(p=0.3),

            # Stage 4: 4x4 -> 1x1
            nn.Conv2d(256, 512, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Dropout(p=0.4),
        )
        self.proj = nn.Sequential(
            nn.Linear(512, embed_dim),
            nn.BatchNorm1d(embed_dim),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x, return_embedding=False):
        f = self.features(x).flatten(1)
        feat = self.proj(f)
        logits = self.classifier(feat)
        if return_embedding:
            return logits, feat
        return logits

model = VGGCIFAR().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"VGG-BN initialized: {total_params:,} parameters")


## 3. Training & Validation


In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS, eta_min=1e-5)

def evaluate(dl):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss_sum += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
    return loss_sum / total, correct / total

best_monitor_acc = 0.0
checkpoint_path = "best_cnn_checkpoint.pt"

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        opt.step()

        tr_loss += loss.item() * len(yb)
        tr_correct += (logits.argmax(1) == yb).sum().item()
        tr_total += len(yb)

    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    train_acc = tr_correct / tr_total
    train_loss = tr_loss / tr_total

    mo_loss, mo_acc = evaluate(monitor_dl)

    if mo_acc > best_monitor_acc:
        best_monitor_acc = mo_acc
        torch.save(model.state_dict(), checkpoint_path)
        saved = "*"
    else:
        saved = ""

    if epoch % 5 == 0 or epoch == 1 or epoch == NUM_EPOCHS or saved != "":
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS:02d} | LR: {current_lr:.6f} | Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | Monitor Acc: {mo_acc:.3f} (Best: {best_monitor_acc:.3f}) {saved}")

print(f"Training finished. Best monitor accuracy: {best_monitor_acc:.4f}")


## 4. Feature Extraction & Export


In [ ]:
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    print(f"Loaded checkpoint from {checkpoint_path}")

@torch.no_grad()
def export_split(dl):
    model.eval()
    embs, probs, preds, labels, correct = [], [], [], [], []
    for xb, yb in dl:
        xb = xb.to(device)
        logits, feat = model(xb, return_embedding=True)
        pr = F.softmax(logits, dim=1).cpu()
        p = pr.argmax(1)
        embs.append(feat.cpu())
        probs.append(pr)
        preds.append(p)
        labels.append(yb.cpu())
        correct.append((p == yb.cpu()).int())
    return (
        torch.cat(embs),
        torch.cat(probs),
        torch.cat(preds),
        torch.cat(labels),
        torch.cat(correct),
    )

# Validation split (used for RL calibration)
val_emb, val_probs, val_pred, val_label, val_correct = export_split(val_dl)
torch.save({
    "embedding": val_emb,
    "probs": val_probs,
    "pred": val_pred,
    "label": val_label,
    "correct": val_correct,
}, "cnn_val_export.pt")
print("Saved cnn_val_export.pt:", val_emb.shape, "acc:", f"{val_correct.float().mean().item():.4f}")

# Test split (used for final evaluation)
test_emb, test_probs, test_pred, test_label, test_correct = export_split(test_dl)
torch.save({
    "embedding": test_emb,
    "probs": test_probs,
    "pred": test_pred,
    "label": test_label,
    "correct": test_correct,
}, "cnn_test_export.pt")
print("Saved cnn_test_export.pt:", test_emb.shape, "acc:", f"{test_correct.float().mean().item():.4f}")
